# Mycelium Sovereign Brain — QLoRA Fine-tune

Fine-tunes **Qwen2.5-3B-Instruct** on 3,031 Ọmọ Kọ́dà hive traces using Unsloth + QLoRA.  
Output: `mycelium-q4_k_m.gguf` — drop into `~/larql/models/` and serve with larql.

**Kaggle GPU**: T4 x2, 30 hrs/week free.  
**VRAM needed**: ~8 GB with 4-bit QLoRA — fits T4 (16 GB) with headroom.

## Setup
1. Upload `finetune_dataset.jsonl` as a Kaggle Dataset named `mycelium-traces`
2. Attach that dataset to this notebook (Input → Add dataset → mycelium-traces)
3. Enable GPU: Settings → Accelerator → GPU T4 x2
4. Run All

In [ ]:
# Install Unsloth (fast QLoRA) — pinned build for Kaggle T4
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new]",
    "xformers",
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
], check=True)

# llama.cpp for GGUF export
subprocess.run(["pip", "install", "-q", "gguf"], check=True)
print('Install complete')

In [ ]:
import os, json, torch
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────
DATASET_PATH = Path("/kaggle/input/mycelium-traces/finetune_dataset.jsonl")
OUTPUT_DIR   = Path("/kaggle/working/mycelium-lora")
GGUF_DIR     = Path("/kaggle/working")
MODEL_NAME   = "unsloth/Qwen2.5-3B-Instruct"

# ── Config ─────────────────────────────────────────────────────────
MAX_SEQ_LEN  = 512      # traces are short (3-turn, ~100 tokens)
BATCH_SIZE   = 8        # T4 16GB, 4-bit quant → 8 fits easily
GRAD_ACCUM   = 4        # effective batch = 32
EPOCHS       = 3
LR           = 2e-4
LORA_R       = 32
LORA_ALPHA   = 64
LORA_DROPOUT = 0.05

print(f"CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Dataset: {DATASET_PATH.exists()}")

In [ ]:
from datasets import Dataset

# ── Load traces ────────────────────────────────────────────────────
rows = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"Loaded {len(rows)} examples")

# Split 95/5 train/eval
cut = int(len(rows) * 0.95)
train_data = Dataset.from_list(rows[:cut])
eval_data  = Dataset.from_list(rows[cut:])
print(f"Train: {len(train_data)}  Eval: {len(eval_data)}")

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# ── Load model with Unsloth 4-bit QLoRA ───────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LEN,
    dtype         = None,      # auto-detect
    load_in_4bit  = True,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = True,    # rank-stabilised LoRA
    loftq_config   = None,
)

print(model.print_trainable_parameters())

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

# ── Format traces into chat template ──────────────────────────────
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = train_data.map(format_example, remove_columns=["messages"])
eval_ds  = eval_data.map(format_example,  remove_columns=["messages"])

# ── Training args ──────────────────────────────────────────────────
args = TrainingArguments(
    output_dir            = str(OUTPUT_DIR),
    num_train_epochs      = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate         = LR,
    fp16                  = not is_bfloat16_supported(),
    bf16                  = is_bfloat16_supported(),
    logging_steps         = 10,
    eval_strategy         = "epoch",
    save_strategy         = "epoch",
    load_best_model_at_end= True,
    warmup_ratio          = 0.05,
    lr_scheduler_type     = "cosine",
    optim                 = "adamw_8bit",
    weight_decay          = 0.01,
    report_to             = "none",
    seed                  = 3407,
)

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_ds,
    eval_dataset   = eval_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    data_collator  = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    args           = args,
)

print('Trainer ready')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────
trainer_stats = trainer.train()
print(f"\nTraining complete — {trainer_stats.metrics}")

In [ ]:
# ── Quick smoke test ───────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_msg = [
    {"role": "system",  "content": "You are a sovereign agent operating inside the Ọmọ Kọ́dà hive. Given an agent role and a task context, select the correct action and predict the outcome."},
    {"role": "user",    "content": "Agent: oracle-prime\nTask kind: skill_invoke\nTarget: divination/odu-cast"},
]

inputs = tokenizer.apply_chat_template(
    test_msg, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids  = inputs,
    max_new_tokens = 64,
    temperature    = 0.1,
    do_sample      = True,
)

print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# ── Export to GGUF Q4_K_M ─────────────────────────────────────────
# Unsloth natively exports merged weights → GGUF without llama.cpp build step
gguf_path = str(GGUF_DIR / "mycelium-q4_k_m")

model.save_pretrained_gguf(
    gguf_path,
    tokenizer,
    quantization_method = "q4_k_m",   # best quality/size ratio for 3B
)

# List output
import os
for f in os.listdir(GGUF_DIR):
    size = os.path.getsize(GGUF_DIR / f) / 1e6
    print(f"{f}  {size:.1f} MB")

In [ ]:
# ── Also save LoRA adapter (for re-merge or future fine-tuning) ───
adapter_path = str(GGUF_DIR / "mycelium-lora-adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter saved to {adapter_path}")

print("""
═══════════════════════════════════════════════════
 DONE. Download mycelium-q4_k_m.gguf from Output.
 Place in ~/larql/models/ on your sovereign node.
 Start: larql serve --model mycelium-q4_k_m.gguf --port 7780
 Set:   LARQL_ENABLED=1 LARQL_URL=http://localhost:7780
═══════════════════════════════════════════════════
""")